# Chapter 1: Basic Prompt Structure

**Technique:** Prompt anatomy with system, user, and assistant roles

**Contents**
* [Lesson: The Messages API](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [1]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: The Messages API

Every Claude API call requires three top level fields:

* `model` (`str`): pinned model ID (e.g. `claude-opus-5-5`)
* `max_tokens` (`int`): hard upper bound on output tokens
* `messages` (`list`): ordered list of conversation turns

Each element of `messages` is a dict `{"role": ..., "content": ...}` where `role` is either `"user"` or `"assistant"`. A well-formed conversation follows two conventions:

1. **The first message is from the `user`.**
2. **Roles alternate**: `user`, `assistant`, `user`, ... one turn at a time.

The API itself enforces less than that, and it pays to know exactly what:

* **Every message needs a `role`.** A message without one is rejected with a `400`.
* **The conversation must end with a `user` turn** (on current models). A trailing `assistant` turn, the old "prefill" pattern, is rejected with a `400`.
* **Consecutive turns with the same role are merged**, and a leading `assistant` turn is accepted. These no longer fail loudly, which is exactly why you should get them right in your own code: a bug in your conversation-state logic silently changes what Claude sees instead of raising an error.

The `system` prompt is a separate top level string, not a message turn. Use it to set Claude's persona, constraints, or output format. The helper `get_completion(prompt, system="")` wires all of this up; `system` defaults to `""` (no system prompt).

When you call `client.messages.create` directly, as some cells below do, we also pass `**SAMPLING_PARAMS`. It adds `temperature=0` on older models that accept it and nothing on current models (see Chapter 0).

This is the same shape you'll wire into a backend service: construct a `messages` list from your application's conversation state, pass it to the API, stream or wait for the reply.

In [2]:
print(get_completion("What is the time complexity of binary search?"))

Binary search has a time complexity of **O(log n)**.

## Why O(log n)?

Each comparison eliminates **half** of the remaining elements:

| Step | Elements remaining |
|------|-------------------|
| 0 | n |
| 1 | n/2 |
| 2 | n/4 |
| k | n/2ᵏ |

The search ends when one element remains, so n/2ᵏ = 1, which gives **k = log₂ n**.

For example, searching 1,000,000 elements takes at most about **20 comparisons**.

## Breakdown by Case

| Case | Time Complexity |
|------|----------------|
| Best case | **O(1)**: the target is the middle element |
| Average case | **O(log n)** |
| Worst case | **O(log n)** |

## Space Complexity

- **Iterative:** O(1)
- **Recursive:** O(log n), because of the call stack

## Important Requirement

Binary search only works on **sorted** data with **random access**, such as arrays. If the data must be sorted first, that adds O(n log n). On a linked list, reaching the middle element takes linear time, so binary search loses its advantage.


In [3]:
print(get_completion("What year was Python first released?"))

Python was first released in **February 1991**, when Guido van Rossum published version 0.9.0. He had begun working on it in the late 1980s. Python 1.0 followed in January 1994.


### Malformed requests (intentional errors)

The cells below send requests that break the message rules. Two of them violate the API contract and **are rejected with a `400 Bad Request`**; we catch the `anthropic.BadRequestError` and print it so you can read exactly what the API enforces (and so "Run All" keeps going). The middle one shows how the API treats consecutive turns with the same role.

In [4]:
import anthropic

try:
    res = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"content": "Hi Claude, how are you?"}],  # missing role: the API rejects this
        **SAMPLING_PARAMS,
    )
    print(response_text(res))
except anthropic.BadRequestError as err:
    print("400 Bad Request:", err.message)

400 Bad Request: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.role: Field required'}, 'request_id': 'req_011CfrQYhnXzMyTvwdRXMKmD'}


In [5]:
# Two consecutive "user" turns. Older versions of the API rejected this with a 400;
# the current API merges consecutive same-role messages into a single user turn,
# so Claude answers both questions. Build alternating turns in your own code anyway.
try:
    res = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": "What is REST?"},
            {"role": "user", "content": "Also, what is gRPC?"},
        ],
        **SAMPLING_PARAMS,
    )
    print(response_text(res))
except anthropic.BadRequestError as err:
    print("400 Bad Request:", err.message)

# REST

**REST (Representational State Transfer)** is an architectural style for designing networked APIs, usually over HTTP. Roy Fielding defined it in his 2000 doctoral dissertation.

### Core ideas
- **Resources**: Everything is a resource (a user, an order, a product), identified by a URL.
  - `https://api.example.com/users/42`
- **HTTP methods** express actions on resources:
  | Method | Typical meaning |
  |---|---|
  | `GET` | Read a resource |
  | `POST` | Create a resource |
  | `PUT` / `PATCH` | Replace / partially update |
  | `DELETE` | Remove |
- **Statelessness**: Each request carries all the information needed to process it. The server doesn't keep client session state between requests.
- **Representations**: Resources are sent in some format, most commonly **JSON** (sometimes XML).
- **Standard HTTP status codes**: `200 OK`, `201 Created`, `404 Not Found`, `500 Internal Server Error`, etc.
- **Cacheability**: Responses can be marked cacheable using HTTP caching mechanis

In [6]:
# A conversation that ends with an assistant turn (a "prefill") is rejected on
# current models: the last message must be the user turn you want answered.
try:
    res = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": "Classify this ticket: 'I was charged twice this month.'"},
            {"role": "assistant", "content": "Category:"},  # trailing assistant turn
        ],
        **SAMPLING_PARAMS,
    )
    print(response_text(res))
except anthropic.BadRequestError as err:
    print("400 Bad Request:", err.message)

400 Bad Request: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011CfrQZvFA4Y17v8mx3FFMT'}


### System prompts

The `system` prompt is delivered to Claude before any user turn. Use it to supply context and rules that should govern the entire conversation: persona, output format, domain constraints, safety guardrails. The user turn then carries the actual request.

The example below configures Claude as a requirements reviewer that only asks clarifying questions, never proposes solutions. This pattern is useful for guardrailing Claude to a specific workflow role inside a larger application.

In [7]:
SYSTEM_PROMPT = "You are a senior engineer doing requirements review. Respond ONLY with clarifying questions that surface missing requirements. Do not propose a solution."
print(get_completion("Build me a rate limiter.", SYSTEM_PROMPT))

## Purpose & Scope
1. What problem is this solving: abuse prevention, protecting a fragile downstream service, cost control, fair usage across tenants, or contractual/tiered API quotas?
2. What are we limiting: inbound requests to our API, outbound calls to a third party, background jobs, or something else (messages, logins, bytes)?
3. Is this a library embedded in one service, shared middleware, a standalone service, or config on an existing gateway or proxy (e.g., Envoy, NGINX, API Gateway)? Have existing off-the-shelf options already been considered and ruled out?

## Identity & Granularity
4. What is the limiting key: user ID, API key, tenant/org, IP address, endpoint, or a combination?
5. How should unauthenticated traffic be identified and limited?
6. Do different endpoints or operations have different costs? Should a heavy query consume more quota than a lightweight read?
7. Are limits hierarchical, such as per-user limits nested inside a per-tenant limit inside a global limit?


<a id="exercises"></a>
## Exercises

### Exercise 1.1: Count to three

**Task**: Edit `PROMPT` so that Claude's response contains the numbers 1, 2, and 3.

**Success criteria**: the response text includes all of `"1"`, `"2"`, and `"3"`.

In [8]:
from lib.grading import includes_all, grade

PROMPT = "Count from 1 to 3, listing each number."

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_all(text, ["1", "2", "3"])

grade(response, grade_exercise(response))

1
2
3

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


In [9]:
from hints import exercise_1_1_hint
print(exercise_1_1_hint)

The grader checks that the response text contains the characters "1", "2", and "3". You can usually get Claude to do exactly what you want just by asking plainly.


### Exercise 1.2: Terse TODO answers

**Scenario**: you're building a CLI tool that surfaces quick engineering guidance. You want answers that are terse and action oriented, formatted as a `# TODO` style code comment, not prose.

**Task**: Edit `SYSTEM_PROMPT` so that Claude answers as a terse senior engineer who replies only with a `# TODO` style code comment.

**Success criteria**: the response contains `"TODO"`.

In [10]:
from lib.grading import includes_any, grade

SYSTEM_PROMPT = "You are a terse senior engineer. Reply ONLY with a single-line # TODO: code comment giving the action to take. No prose, no explanation."
PROMPT = "How should I handle a failed database connection on startup?"

response = get_completion(PROMPT, SYSTEM_PROMPT)

def grade_exercise(text):
    return includes_any(text, ["TODO"])

grade(response, grade_exercise(response))

# TODO: Retry the DB connection with exponential backoff and jitter (e.g., 5 attempts, capped delay), log each failure with context, then fail fast with a clear error and non-zero exit so the orchestrator can restart.

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


In [11]:
from hints import exercise_1_2_hint
print(exercise_1_2_hint)

The grader looks for a "TODO" marker. Use the system prompt to make Claude answer as a terse senior engineer who replies only with a # TODO style code comment.


## Common mistakes

* **Ending on an `"assistant"` turn**: on current models the API rejects a `messages` list whose last element has `"role": "assistant"` (the old prefill pattern). Always end with the user turn you want answered.
* **Sloppy conversation state**: starting with an `assistant` turn or appending two `user` turns in a row no longer raises an error, because the API tolerates and merges them. That makes these bugs silent. Keep your history strictly alternating, starting with `user`.
* **Putting constraints in the user turn instead of `system`**: rules buried in the user message can be overridden or forgotten partway through the conversation. Stable constraints belong in the system prompt where they persist across all turns.

**Reflect**: when would you put a constraint in the system prompt vs the user message? Consider what changes from conversation to conversation vs what is fixed for all users of your application.

<a id="example-playground"></a>
## Example Playground

Use the cell below to experiment freely. Swap in any prompt. The graders are not active here.

In [12]:
print(get_completion("Explain the CAP theorem in two sentences."))

The CAP theorem states that a distributed data system cannot simultaneously guarantee Consistency (every read sees the most recent write), Availability (every request receives a non-error response), and Partition tolerance (the system keeps operating despite network failures between nodes). Since network partitions are unavoidable in practice, the theorem means that when a partition occurs, a system must choose between staying consistent by rejecting some requests and staying available by possibly returning stale data.
